# LTX-2.3 ICLoRA LipDub — ComfyUI

Kaynak videonun dudak hareketini ve pose'unu koruyarak, text prompt ile tarif edilen yeni bir karakterde video üret.

## Pipeline
```
Input Video (lip sync kaynağı) → IC-LoRA Guide (hareket + dudak yapısını koru) → Text Prompt (hedef karakter tarifi) → LTX 2.3 22B → Yeni Karakter Video
```

## Main Model
**LTX 2.3 22B Dev** + IC-LoRA Union Control + Distilled LoRA

## Nasıl Çalışır
- **Kaynak video:** Dudak hareket eden kişinin videosu (hareket + lip sync buradan alınır)
- **Hedef karakter:** Referans görsel yok — sadece **text prompt** ile tarif edilir
- Örnek prompt: *"anime girl with blue hair"*, *"old man with white beard"*
- IC-LoRA kaynak videonun yapısını korur, prompt'a göre görünümü değiştirir

## Colab Secrets
- `CF_TUNNEL_TOKEN` — Cloudflare tunnel
- `HF_TOKEN` — HuggingFace

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat

## Custom Node'lar
| Node | Kaynak | Kullanım |
|------|--------|----------|
| ComfyUI-LTXVideo | Lightricks | LTX model loader, audio VAE, IC-LoRA guide, sampler |
| ComfyUI-KJNodes | kijai | Image/video utility |

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# ComfyUI
COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

# Custom Node'lar
NODES = {
    'ComfyUI-LTXVideo': 'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
}

!pip install -q -U --pre comfyui-manager

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# kornia uyumluluk fix (ComfyUI-LTXVideo, pyramid_blending.py 'pad' import hatası)
!pip install -q kornia==0.7.3

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

LTX 2.3 22B Dev + IC-LoRA Union Control + Distilled LoRA + Audio VAE + Upscaler

In [ ]:
import shutil

from huggingface_hub import hf_hub_download

MODELS_DIR = f'{COMFY_DIR}/models'

def hf_download(repo, filename, dest_dir):
    """HuggingFace'ten dosya indir. Mevcutsa atla."""
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Ba\u015far\u0131s\u0131z: {e}')

# === Checkpoint (Diffusion Model) ===
# Workflow 'workflow_setup/40a2610d-.../ltx-2.3-22b-dev.safetensors' path'inde arıyor
print('\U0001f4e5 LTX 2.3 22B Dev Checkpoint:')
SETUP_DIR = f'{MODELS_DIR}/checkpoints/workflow_setup/40a2610d-fabe-4286-aa62-20b508e04d49'
os.makedirs(SETUP_DIR, exist_ok=True)
hf_download(
    'Lightricks/LTX-2.3',
    'ltx-2.3-22b-dev.safetensors',
    SETUP_DIR
)

# === Text Encoder ===
# LTXAVTextEncoderLoader da ayni checkpoint'u arıyor (text enc içinde)
# Ama ayrica comfy_gemma_3_12B_it.safetensors istiyor
print('\n\U0001f4e5 Text Encoder (Gemma 3 12B):')
hf_download(
    'Comfy-Org/ltx-2',
    'split_files/text_encoders/gemma_3_12B_it.safetensors',
    f'{MODELS_DIR}/text_encoders'
)
# Workflow 'comfy_gemma_3_12B_it.safetensors' adıyla arıyor
te_src = f'{MODELS_DIR}/text_encoders/gemma_3_12B_it.safetensors'
te_link = f'{MODELS_DIR}/text_encoders/comfy_gemma_3_12B_it.safetensors'
if os.path.exists(te_src) and not os.path.exists(te_link):
    os.symlink(te_src, te_link)
    print('  \u2705 symlink: comfy_gemma_3_12B_it -> gemma_3_12B_it')

# === Audio VAE ===
# LTXVAudioVAELoader -> models/vae/ klasöründe arar
# Kijai reposundan (metadata içerir, ComfyUI uyumlu)
print('\n\U0001f4e5 Audio VAE:')
hf_download(
    'Kijai/LTX2.3_comfy',
    'vae/LTX23_audio_vae_bf16.safetensors',
    f'{MODELS_DIR}/vae'
)
# Workflow 'ltx-2-3-22b-audio_vae.safetensors' adıyla arıyor
vae_src = f'{MODELS_DIR}/vae/LTX23_audio_vae_bf16.safetensors'
vae_link = f'{MODELS_DIR}/vae/ltx-2-3-22b-audio_vae.safetensors'
if os.path.exists(vae_src) and not os.path.exists(vae_link):
    os.symlink(vae_src, vae_link)
    print('  \u2705 symlink: ltx-2-3-22b-audio_vae -> LTX23_audio_vae_bf16')

# === LoRA'lar ===
print('\n\U0001f4e5 LoRA\'lar:')
# IC-LoRA Union Control (lip sync + referans)
# Workflow 'ltxv/ltx2/' subpath'inde arıyor
LORA_DIR = f'{MODELS_DIR}/loras/ltxv/ltx2'
os.makedirs(LORA_DIR, exist_ok=True)
hf_download(
    'Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control',
    'ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors',
    LORA_DIR
)
# Distilled LoRA - workflow_setup path'inde
LORA_SETUP_DIR = f'{MODELS_DIR}/loras/workflow_setup/40a2610d-fabe-4286-aa62-20b508e04d49/ltxv/ltx2'
os.makedirs(LORA_SETUP_DIR, exist_ok=True)
hf_download(
    'Lightricks/LTX-2.3',
    'ltx-2.3-22b-distilled-lora-384-1.1.safetensors',
    LORA_SETUP_DIR
)

# === Spatial Upscaler ===
# LatentUpscaleModelLoader -> 'latent_upscale_models' klasörüne bakar
print('\n\U0001f4e5 Spatial Upscaler:')
hf_download(
    'Lightricks/LTX-2.3',
    'ltx-2.3-spatial-upscaler-x2-1.1.safetensors',
    f'{MODELS_DIR}/latent_upscale_models'
)

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy

`USE_CLOUDFLARE = True`: Cloudflare tunnel — `comfyui.ersamely.com`

## Workflow Yükleme
1. ComfyUI açılınca **Load** → `ComfyUI LTX-2.3 ICLoRA LipDub Workflow.json`
2. **LoadVideo** node'una kaynak video yükle (dudak hareket eden kişi)
3. **Positive Prompt**'a hedef karakteri yaz (orn: *anime girl with blue hair*)
4. **Queue Prompt** → Aynı dudak hareketleriyle yeni karakterde video üretir

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False

PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*', '--enable-manager'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI \u00e7\u00f6kt\u00fc!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)